# <mark style="display:block; background:#d1c4e9; color:#1a1a1a; padding:6px 12px; border-radius:4px">10/2(금) 오후 · 트리거 · 멱등성 · 스케줄러 — 실습</mark>

오전에 **받는 문**을 만들었습니다. 오후는 **그 문으로 스스로 알림을 보내는 쪽**을 만듭니다.

「어제 새벽 3시에 admin 을 노린 로그인이 또 있었는데, 우리 프로그램은 뭘 하고 있었지?」
— 아무것도 안 했습니다. **사람이 실행 버튼을 눌러야만** 돌기 때문입니다.

오늘 오후의 도착점은 **`scheduler_job.py`** 입니다. 부르지 않아도 스스로 돕니다.


## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">시작하기</mark>

### 0.1 맨 먼저 · 내 사본 만들기

위 메뉴에서 파일 › 드라이브에 사본 저장을 누릅니다.

### 0.2 오늘 오후의 순서

| 교시 | 무엇 |
|---|---|
| 5교시 | 트리거·조건·액션 — 지금까지 만든 게 사실 한 모양이었다 |
| 6교시 | 멱등성 — 같은 알림이 세 번 가면 아무도 안 본다 |
| 7교시 | `schedule` · `cron` · `scheduler_job.py` 조립 |

### 0.3 준비

아래 셀들을 차례로 실행합니다. 오전 서버와 9/29 결과 파일을 여기서 다시 만듭니다.


In [ ]:
!pip install -q schedule


In [ ]:
%%writefile raw_logs.txt
2026-09-29 09:02:11 INFO accepted login for kim.cs from 10.1.2.11
2026-09-29 09:12:00 WARN failed login for kim01 from 203.0.113.5
2026-09-29 09:15:31 INFO accepted login for lee.yh from 10.1.2.34
2026-09-29 09:16:02 INFO session closed for 10.1.2.34
2026-09-29 10:03:19 WARN failed login for park.js from 10.1.3.7
2026-09-29 10:03:31 INFO accepted login for park.js from 10.1.3.7
2026-09-29 11:20:55 INFO accepted login for choi.mk from 10.1.4.2
2026-09-29 12:40:12 INFO session closed for 10.1.4.2
2026-09-29 03:11:05 WARN failed login for admin from 211.45.12.9
2026-09-29 03:12:47 WARN failed login for admin from 211.45.12.9
2026-09-29 03:13:58 WARN failed login for admin from 211.45.12.9
2026-09-29 03:15:22 WARN failed login for admin from 211.45.12.9
2026-09-29 03:17:09 INFO accepted login for admin from 211.45.12.9
2026-09-29 14:05:38 INFO accepted login for jung.hw from 10.1.2.88
2026-09-29 22:14:03 WARN failed login for kim.cs from 185.220.101.34
2026-09-29 22:14:21 WARN failed login for lee.yh from 185.220.101.34
2026-09-29 22:14:40 WARN failed login for choi.mk from 185.220.101.34
2026-09-29 22:14:58 WARN failed login for jung.hw from 185.220.101.34
2026-09-29 22:15:12 INFO session closed for 185.220.101.34
2026-09-29 23:40:07 WARN failed login for invalid user guest from 185.220.101.34


In [ ]:
import re
import json

PATTERN = r"(?P<time>\d{2}:\d{2}:\d{2}) (?P<level>\w+) \w+ login for (?P<user>[\w.]+) from (?P<ip>[\d.]+)"

rows = []
with open("raw_logs.txt", encoding="utf-8") as f:
    for line in f:
        m = re.search(PATTERN, line)
        if m:
            rows.append(m.groupdict())

with open("normalized_logs.json", "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)

print(f"정규화 {len(rows)}건 — 9/29 결과를 다시 만들었습니다")


In [ ]:
%%writefile webhook_server.py
import argparse
import json
from flask import Flask, request

parser = argparse.ArgumentParser(description="경보를 받는 웹훅 서버")
parser.add_argument("--port", type=int, default=5000)
args = parser.parse_args()

app = Flask(__name__)
received = []


@app.route("/webhook", methods=["POST"])
def webhook():
    event = request.get_json()
    received.append(event)
    with open("received_alerts.json", "w", encoding="utf-8") as f:
        json.dump(received, f, ensure_ascii=False, indent=2)
    return {"status": "ok", "count": len(received)}, 200


app.run(port=args.port)


In [ ]:
import subprocess
import sys
import time


def start_server(path, port):
    """서버 파일을 백그라운드로 띄웁니다. 셀이 멈추지 않습니다."""
    proc = subprocess.Popen([sys.executable, path, "--port", str(port)])
    time.sleep(2)
    print(f"{path} 를 {port} 번 포트에 띄웠습니다")
    return proc


server = start_server("webhook_server.py", 5005)


---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">5교시 (14:00–14:50) · 트리거 · 조건 · 액션</mark>


## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다. 검색이든 공식 문서든 AI 든 상관없습니다. 찾은 것은 코랩에서 한 번 실행해 확인합니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **워크플로(workflow)** | 일이 흘러가는 순서를 왜 정해 두나 |
| **트리거(trigger)** | 일을 시작하게 만드는 것은 무엇인가 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다. 한 줄씩이면 됩니다.

- 워크플로(workflow) →
- 트리거(trigger) →

맨 처음에 「드라이브에 사본 저장」을 눌렀다면 여기 적은 것은 내 드라이브의 노트북에 함께 남습니다.

적어 둔 것은 이 교시가 끝날 때 다시 봅니다. 찾은 것과 배운 것이 어디서 갈렸는지가 오늘의 공부입니다.


---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">4 · 지금까지 만든 게 사실 한 모양이었다</mark>


### 왜 필요한가

1. 나흘 동안 만든 것을 늘어놓으면 겉모습이 다 다릅니다. 파일을 읽는 것, API 를 부르는 것, 웹훅을 받는 것.
2. 그런데 **세 조각으로 쪼개 보면 전부 같은 모양**입니다.
3. **무엇이 시작을 알렸는가**(트리거) · **어떤 경우에만 움직이는가**(조건) · **그래서 무엇을 하는가**(액션).


### 나흘치를 한 표에 놓으면

| 트리거 | 조건 | 액션 | 언제 만들었나 |
|---|---|---|---|
| 파일이 있다 | 깨진 줄인가 | 건너뛰고 기록 | 9/28 `log_parser.py` |
| 로그가 쌓였다 | 실패 3회 이상인가 | 경보를 찍는다 | 9/29 룰 ① |
| 경보가 났다 | 처음 보는 IP 인가 | 조회한다 | 9/30 `api_client.py` |
| **시각이 됐다** | **처음 보는 사건인가** | **알린다** | **오늘** |

거창한 것이 아닙니다. **「이런 모양이면 이렇게 하라」**를 코드로 적은 것뿐입니다.


### 이 시간에 나오는 말

| 말 | 뜻 |
|---|---|
| 워크플로 | 일이 흘러가는 정해진 순서 |
| 트리거 | 일을 시작하게 만드는 사건 |
| 조건 | 그중 어떤 경우에만 움직일지 |
| 액션 | 그래서 실제로 하는 일 |


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.1 세 조각으로 갈라 적는다</mark>

9/29에 쓴 룰 ① 코드를 세 조각으로 갈라 보면 이렇습니다.

```python
for row in rows:                          # ← 트리거 : 로그가 있다
    if row["level"] == "WARN":            # ← 조건 : 실패인가
        count[row["user"]] = ...          # ← 액션 : 센다
```

갈라 적으면 **바꿀 곳이 분명해집니다.** 기준을 바꾸려면 조건만, 알림 방식을 바꾸려면 액션만 고칩니다.


아래 셀을 먼저 실행합니다. 오늘 오후 내내 이 `events` 를 씁니다. 9/29 룰 세 개가 찾아낸 경보입니다.


In [ ]:
events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

print(f"경보 {len(events)}건")


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

print(events[0]["rule"])
```

막히면 바로 위 `4.1 세 조각으로 갈라 적는다` 설명을 다시 봅니다.


In [ ]:
events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

print(events[0]["rule"])


✅ `brute_force`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-2 · 무엇이 보일까요</font></h3></td></tr></table>

이번에는 조건을 하나 붙입니다.

```python
events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

for event in events:
    if event["rule"] == "night_login":
        print(event["time"])
```

막히면 바로 위 `4.1 세 조각으로 갈라 적는다` 설명을 다시 봅니다.


In [ ]:
events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

for event in events:
    if event["rule"] == "night_login":
        print(event["time"])


✅ `03:17:09`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-3 · 경보 이름만 훑기</font></h3></td></tr></table>

`events` 를 돌면서 **룰 이름**을 한 줄씩 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `brute_force` · `password_spraying` · `night_login` 세 줄 |

**💡 힌트**

1. `for event in events:` 로 하나씩 꺼냅니다.
2. 꺼낸 것은 딕셔너리라 이름으로 꺼냅니다.
3. 출력은 반복 안에 둡니다.


In [ ]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-4 · 조건을 하나 붙이기</font></h3></td></tr></table>

`night_login` **이 아닌** 경보만 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `brute_force` · `password_spraying` 두 줄 |

**💡 힌트**

1. 같지 않은지는 `!=` 로 견줍니다.
2. `if` 를 반복 안에 둡니다.
3. 이 줄이 **조건** 조각입니다.


In [ ]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-5 · 세 조각에 주석 달기</font></h3></td></tr></table>

아래 코드에서 **트리거·조건·액션**이 어느 줄인지 주석으로 표시하시오.

```python
for event in events:
    if event["rule"] == "brute_force":
        print("[알림]", event["user"])
```

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[알림] admin` 과 주석 세 줄 |

**💡 힌트**

1. 반복을 여는 줄이 **트리거**입니다 — 무엇이 시작을 알렸나.
2. `if` 줄이 **조건**입니다 — 어떤 경우에만 움직이나.
3. 안쪽 줄이 **액션**입니다 — 그래서 무엇을 하나.


In [ ]:
# 여기에 코드를 입력하세요


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 4-1 · 조건만 바꿔 보기</font></h3></td></tr></table>

위 코드에서 **조건 줄 하나만** 바꿔 `night_login` 경보가 나오게 하시오. 나머지 줄은 손대지 않습니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[알림] admin` — 이번에는 심야 접속 때문 |

**💡 힌트**

1. `if` 줄의 값만 바꿉니다.
2. 트리거와 액션은 그대로 둡니다.
3. **갈라 적으면 한 줄만 고치면 된다**는 것이 이 문제의 요점입니다.


In [ ]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.2 액션을 함수로 — 오전에 연 문으로 보낸다</mark>

액션을 함수로 떼어 두면 **알림 방식을 바꿀 때 한 곳만** 고칩니다.
오늘은 그 액션이 **오전에 만든 웹훅 서버로 보내는 것**입니다.

```python
import requests


def send_alert(event):
    response = requests.post("http://127.0.0.1:5005/webhook", json=event, timeout=5)
    return response.json()
```

- `requests.post` 는 9/30에 배운 `get` 의 짝입니다. **보낼 내용은 `json=` 에** 담습니다.
- 준비 셀이 서버를 **5005** 번에 띄워 두었습니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-6 · 무엇이 보일까요</font></h3></td></tr></table>

서버가 살아 있는지 먼저 봅니다.

```python
!curl -s -o /dev/null -w "%{http_code}" -X POST -H "Content-Type: application/json" -d '{"rule":"ping"}' http://127.0.0.1:5005/webhook
```

막히면 바로 위 `4.2 액션을 함수로` 설명을 다시 봅니다.


In [ ]:
!curl -s -o /dev/null -w "%{http_code}" -X POST -H "Content-Type: application/json" -d '{"rule":"ping"}' http://127.0.0.1:5005/webhook


✅ `200`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-7 · 무엇이 보일까요</font></h3></td></tr></table>

이번에는 파이썬에서 보냅니다.

```python
import requests

response = requests.post("http://127.0.0.1:5005/webhook",
                         json={"rule": "brute_force", "user": "admin"}, timeout=5)

print(response.json()["status"])
```

막히면 바로 위 `4.2 액션을 함수로` 설명을 다시 봅니다.


In [ ]:
import requests

response = requests.post("http://127.0.0.1:5005/webhook",
                         json={"rule": "brute_force", "user": "admin"}, timeout=5)

print(response.json()["status"])


✅ `ok`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-8 · `send_alert` 만들기</font></h3></td></tr></table>

경보 하나를 받아 서버로 보내는 **`send_alert`** 함수를 만드시오. 서버가 돌려준 `count` 를 돌려줍니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 숫자 하나 (보낼 때마다 늘어난다) |

**💡 힌트**

1. `def send_alert(event):` 로 시작합니다.
2. `requests.post(주소, json=event, timeout=5)` 로 보냅니다.
3. 돌려받은 것에서 `count` 를 꺼내 `return` 합니다.


In [ ]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-9 · 세 건을 모두 보내기</font></h3></td></tr></table>

`events` 세 건을 모두 보내고, 보낼 때마다 **룰 이름과 결과**를 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[전송] brute_force → ok` 꼴로 세 줄 |

**💡 힌트**

1. 문제 4-8의 함수를 그대로 씁니다. 이번에는 `status` 를 돌려주게 해도 됩니다.
2. `for` 로 세 건을 돕니다.
3. f-string 으로 한 줄씩 냅니다.


In [ ]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-10 · 보낸 것을 서버에서 확인하기</font></h3></td></tr></table>

서버가 남긴 **`received_alerts.json`** 을 읽어 **몇 건 받았는지**와 **룰 이름들**을 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 받은 건수와 룰 이름 목록 |

**💡 힌트**

1. 서버가 받을 때마다 그 파일에 저장합니다.
2. `json.load` 로 읽으면 리스트입니다.
3. `for` 로 돌며 `rule` 을 꺼냅니다.


In [ ]:
# 여기에 코드를 입력하세요


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 4-2 · 실패해도 멈추지 않게</font></h3></td></tr></table>

서버가 꺼져 있을 때도 프로그램이 죽지 않게 `send_alert` 를 고치시오. 9/30에 배운 그대로입니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 꺼진 포트로 보내면 `전송 실패` 가 찍히고 다음 줄이 이어진다 |

**💡 힌트**

1. `try` 안에 `requests.post` 를 둡니다.
2. `except requests.RequestException:` 으로 잡습니다.
3. 실패하면 `None` 을 돌려주고, 부르는 쪽이 `if` 로 확인합니다.


In [ ]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#bbdefb; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.정리 📋 한눈에</mark>

| 조각 | 코드에서 어디 |
|---|---|
| 트리거 | 반복을 여는 줄 — 무엇이 시작을 알렸나 |
| 조건 | `if` 줄 — 어떤 경우에만 움직이나 |
| 액션 | 안쪽 줄 — 그래서 무엇을 하나 |

| 쓰는 법 | 뜻 |
|---|---|
| `requests.post(주소, json=값)` | 보낼 내용을 본문에 담아 보낸다 |
| 액션을 함수로 | 알림 방식을 바꿀 때 **한 곳만** 고친다 |


---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">6교시 (15:00–15:50) · 같은 알림이 세 번 가면</mark>


## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다. 검색이든 공식 문서든 AI 든 상관없습니다. 찾은 것은 코랩에서 한 번 실행해 확인합니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **멱등성(idempotency)** | 같은 일을 두 번 해도 결과가 같다는 것이 왜 중요한가 |
| **중복 알림** | 실무에서 알림이 반복되면 어떤 일이 생기나 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다. 한 줄씩이면 됩니다.

- 멱등성(idempotency) →
- 중복 알림 →

맨 처음에 「드라이브에 사본 저장」을 눌렀다면 여기 적은 것은 내 드라이브의 노트북에 함께 남습니다.

적어 둔 것은 이 교시가 끝날 때 다시 봅니다. 찾은 것과 배운 것이 어디서 갈렸는지가 오늘의 공부입니다.


---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">5 · 이미 보낸 것은 다시 보내지 않는다</mark>


### 왜 필요한가

1. 스케줄러가 5분마다 로그를 읽는다고 해 봅시다. 그런데 **같은 로그가 파일에 그대로** 있습니다.
2. 다음 실행에서도 같은 사건을 보고 **또 알림을 보냅니다.** 하루면 288번입니다.
3. 같은 알림이 세 번 가면 **아무도 안 봅니다.** 진짜 경보가 그 사이에 묻힙니다.
4. 같은 일을 두 번 해도 결과가 한 번 한 것과 같아야 합니다. 이 성질을 **멱등성**이라고 합니다.


### 이 시간에 나오는 말

| 말 | 뜻 |
|---|---|
| 멱등성 | 같은 일을 여러 번 해도 결과가 한 번 한 것과 같은 성질 |
| 사건 번호 | 사건 하나를 가리키는 고유한 이름 |
| `processed_ids.json` | 이미 처리한 사건 번호를 적어 두는 파일 |


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">5.1 중복이 나는 장면을 먼저 본다</mark>

막는 법을 배우기 전에 **실제로 벌어지는 것**을 봅니다.

```python
for turn in [1, 2]:                  # 스케줄러가 두 번 돌았다고 치면
    for event in events:
        print(f"{turn}회차 [전송] {event['rule']}")
```

같은 경보가 **두 번씩** 나갑니다. 5분마다면 하루에 288번입니다.


아래 셀을 먼저 실행합니다. 5교시에서 쓴 `events` 를 다시 만듭니다.


In [ ]:
events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

print(f"경보 {len(events)}건")


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 **몇 줄**이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

sent = 0
for turn in [1, 2]:
    for event in events:
        sent = sent + 1

print("보낸 횟수:", sent)
```

막히면 바로 위 `5.1 중복이 나는 장면을 먼저 본다` 설명을 다시 봅니다.


In [ ]:
events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

sent = 0
for turn in [1, 2]:
    for event in events:
        sent = sent + 1

print("보낸 횟수:", sent)


✅ `보낸 횟수: 6`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-2 · 무엇이 보일까요</font></h3></td></tr></table>

5분마다 하루를 돈다면 몇 번일까요. 하루는 288번 돕니다.

```python
print(3 * 288)
```

막히면 바로 위 `5.1 중복이 나는 장면을 먼저 본다` 설명을 다시 봅니다.


In [ ]:
print(3 * 288)


✅ `864`


경보 세 건이 하루에 **864번** 나갑니다. 받는 사람은 둘째 날부터 안 봅니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-3 · 두 번 돌려 보기</font></h3></td></tr></table>

스케줄러가 **두 번** 돌았다고 치고, 보내는 줄을 화면에 찍으시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `1회차 [전송] …` 세 줄, `2회차 [전송] …` 세 줄 |

**💡 힌트**

1. 바깥 반복이 회차, 안쪽 반복이 경보입니다.
2. 반복 안에 반복을 넣습니다.
3. f-string 에 회차와 룰 이름을 함께 넣습니다.


In [ ]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-4 · 사건 번호를 만든다</font></h3></td></tr></table>

경보마다 **고유한 이름**이 있어야 「이미 보냈다」를 판단할 수 있습니다. `events` 의 `id` 값을 한 줄씩 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `brute_force:admin` 등 세 줄 |

**💡 힌트**

1. `id` 칸이 이미 들어 있습니다.
2. `for` 로 돌며 꺼냅니다.
3. 이 값이 **같은 사건인지 판단하는 열쇠**입니다.


In [ ]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-5 · 이미 보낸 것을 걸러 보기</font></h3></td></tr></table>

이미 보낸 번호를 담은 리스트가 있을 때, **안 보낸 것만** 골라 출력하시오.

| | |
|---|---|
| 주어지는 값 | `done = ["brute_force:admin"]` |
| 🎯 나와야 하는 결과 | `password_spraying:…` 과 `night_login:…` 두 줄 |

**💡 힌트**

1. 없는지 보는 것은 `not in` 입니다.
2. `if event["id"] not in done:` 으로 거릅니다.
3. 이 한 줄이 중복을 막는 핵심입니다.


In [ ]:
done = ["brute_force:admin"]


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 5-1 · 번호를 직접 만들어 보기</font></h3></td></tr></table>

`id` 칸이 없다고 치고, **룰 이름과 계정을 이어 붙여** 번호를 만드시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `brute_force:admin` 꼴 |

**💡 힌트**

1. f-string 으로 두 값을 콜론으로 잇습니다.
2. 계정이 없는 경보는 IP 를 씁니다 — `if "user" in event:` 로 갈라 봅니다.
3. **같은 사건이면 언제나 같은 번호**가 나와야 합니다. 시각을 넣으면 매번 달라질 수 있습니다.


In [ ]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">5.2 `processed_ids.json` — 파일에 적어 둔다</mark>

리스트에만 담아 두면 **프로그램이 꺼지는 순간 사라집니다.** 파일에 남겨야 다음 실행이 압니다.

```python
import json
import os


def load_done():
    if os.path.exists("processed_ids.json"):
        with open("processed_ids.json", encoding="utf-8") as f:
            return json.load(f)
    return []                                  # 처음이면 빈 목록
```

- `os.path.exists` 는 **파일이 있는지** 보는 명령입니다. 처음 돌 때는 파일이 없습니다.
- 저장은 9/28에 배운 `json.dump` 그대로입니다.


아래 셀을 먼저 실행합니다. 읽고 쓰는 함수 두 개를 준비합니다.


In [ ]:
import json
import os


def load_done():
    if os.path.exists("processed_ids.json"):
        with open("processed_ids.json", encoding="utf-8") as f:
            return json.load(f)
    return []


def save_done(done):
    with open("processed_ids.json", "w", encoding="utf-8") as f:
        json.dump(done, f, ensure_ascii=False, indent=2)

print("준비 끝")


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-6 · 무엇이 보일까요</font></h3></td></tr></table>

아직 파일이 없을 때 무엇이 돌아올지 적어 보세요.

```python
import os

if os.path.exists("없는파일.json"):
    print("있다")
else:
    print("없다")
```

막히면 바로 위 `5.2 processed_ids.json — 파일에 적어 둔다` 설명을 다시 봅니다.


In [ ]:
import os

if os.path.exists("없는파일.json"):
    print("있다")
else:
    print("없다")


✅ `없다`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-7 · 무엇이 보일까요</font></h3></td></tr></table>

저장했다가 다시 읽어 봅니다.

```python
import json

with open("demo_ids.json", "w", encoding="utf-8") as f:
    json.dump(["a", "b"], f)

with open("demo_ids.json", encoding="utf-8") as f:
    print(json.load(f))
```

막히면 바로 위 `5.2 processed_ids.json — 파일에 적어 둔다` 설명을 다시 봅니다.


In [ ]:
import json

with open("demo_ids.json", "w", encoding="utf-8") as f:
    json.dump(["a", "b"], f)

with open("demo_ids.json", encoding="utf-8") as f:
    print(json.load(f))


✅ `['a', 'b']`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-8 · 처음 한 번 보내기</font></h3></td></tr></table>

`load_done()` 으로 읽어 **안 보낸 것만** 보내고, 보낸 번호를 파일에 저장하시오. 보낸 건수를 마지막에 출력합니다.

- 실제로 보내지는 말고 화면에 찍기만 합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `보낸 건수 3` |

**💡 힌트**

1. `done = load_done()` 으로 시작합니다.
2. `if event["id"] not in done:` 로 거르고, 보낸 뒤 `done.append` 합니다.
3. 마지막에 `save_done(done)` 을 부릅니다.


In [ ]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-9 · 한 번 더 돌려 보기</font></h3></td></tr></table>

문제 5-8의 코드를 **그대로 한 번 더** 실행하시오. 이번에는 몇 건이 나가는지 봅니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `보낸 건수 0` |

**💡 힌트**

1. 코드를 고치지 않습니다. 같은 셀을 다시 실행하면 됩니다.
2. 앞 실행이 `processed_ids.json` 에 세 건을 적어 두었습니다.
3. 0 이 나오면 **멱등성이 생긴 것**입니다.


In [ ]:
# 문제 5-8 의 코드를 그대로 한 번 더 실행합니다


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-10 · 기록을 열어 보기</font></h3></td></tr></table>

`processed_ids.json` 을 열어 **무엇이 적혀 있는지** 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 사건 번호 세 개가 담긴 리스트 |

**💡 힌트**

1. `load_done()` 을 그대로 쓰면 됩니다.
2. 또는 `!cat processed_ids.json` 으로 파일을 봅니다.
3. 이 파일이 **다음 실행의 기억**입니다.


In [ ]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-11 · 새 경보가 하나 더 오면</font></h3></td></tr></table>

경보 목록에 **새 사건 하나**를 더해 다시 돌리시오. 새 것만 나가야 합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `보낸 건수 1` |

**💡 힌트**

1. `events` 리스트에 딕셔너리 하나를 `append` 합니다.
2. `id` 는 앞의 것들과 달라야 합니다.
3. 나머지 코드는 문제 5-8 그대로입니다.


In [ ]:
# 여기에 코드를 입력하세요


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 5-2 · 기록을 지우면</font></h3></td></tr></table>

`processed_ids.json` 을 **지우고** 다시 돌리면 어떻게 되는지 확인하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 다시 전부 나간다 — 기억이 사라졌기 때문 |

**💡 힌트**

1. 파일을 지우는 명령은 `!rm processed_ids.json` 입니다.
2. 지운 뒤 문제 5-8 을 다시 돌립니다.
3. **기억이 파일에 있다**는 것이 이 문제의 요점입니다. 파일을 잃으면 중복이 다시 납니다.


In [ ]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#bbdefb; color:#1a1a1a; padding:6px 12px; border-radius:4px">5.정리 📋 한눈에</mark>

| 쓰는 법 | 뜻 |
|---|---|
| 사건 번호(`id`) | 같은 사건이면 **언제나 같은 값**이어야 한다 |
| `if event["id"] not in done:` | 이미 보낸 것은 건너뛴다 |
| `os.path.exists(파일)` | 파일이 있는지 본다. 처음엔 없다 |
| `processed_ids.json` | 다음 실행이 읽을 **기억** |

⚠ 기억이 **파일에 있습니다.** 파일을 잃으면 중복이 다시 납니다.


---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">7교시 (16:00–16:50) · 부르지 않아도 스스로 돈다</mark>


## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다. 검색이든 공식 문서든 AI 든 상관없습니다. 찾은 것은 코랩에서 한 번 실행해 확인합니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **스케줄러(scheduler)** | 정해진 시각에 프로그램을 돌리는 방법에는 무엇이 있나 |
| **cron** | 운영체제가 가진 정기 실행 도구. 어떻게 시각을 적나 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다. 한 줄씩이면 됩니다.

- 스케줄러(scheduler) →
- cron →

맨 처음에 「드라이브에 사본 저장」을 눌렀다면 여기 적은 것은 내 드라이브의 노트북에 함께 남습니다.

적어 둔 것은 이 교시가 끝날 때 다시 봅니다. 찾은 것과 배운 것이 어디서 갈렸는지가 오늘의 공부입니다.


---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">6 · 실행 버튼을 없앤다</mark>


### 왜 필요한가

1. 지금까지 만든 것은 **사람이 눌러야** 돕니다. 새벽 3시에 누를 사람이 없습니다.
2. 방법이 둘입니다. **파이썬 안에서 도는 것**(`schedule`)과 **컴퓨터에게 맡기는 것**(`cron`).
3. 둘의 차이는 하나입니다 — `schedule` 은 **그 프로그램이 떠 있어야** 돌고, `cron` 은 **꺼져 있어도** 운영체제가 깨웁니다.


### 이 시간에 나오는 말

| 말 | 뜻 |
|---|---|
| 스케줄러 | 정해진 때에 일을 시키는 것 |
| `schedule` | 파이썬 코드로 간격을 적는 패키지 |
| `cron` | 운영체제가 가진 정기 실행 도구 |
| cron 표현식 | 분·시·일·월·요일 **다섯 칸** |


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">6.1 `schedule` — 파이썬 안에서 돈다</mark>

```python
import schedule
import time


def job():
    print("점검합니다")


schedule.every(2).seconds.do(job)      # 2초마다 job 을 부른다

for _ in range(6):                     # 여섯 번 확인하는 동안만
    schedule.run_pending()
    time.sleep(1)
```

- `schedule.every(10).minutes.do(job)` 처럼 **읽기 쉽게** 적습니다.
- **`run_pending()` 을 계속 불러 줘야** 합니다. 혼자 도는 게 아닙니다.
- 실무에서는 `while True:` 로 무한히 돕니다. 여기서는 **셀이 끝나야 하니 정해진 횟수만** 돕니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-1 · 무엇이 보일까요</font></h3></td></tr></table>

간격을 **10분**으로 두고 몇 초만 돌려 봅니다. 화면에 무엇이 보일지 적어 보세요.

```python
import schedule
import time


def job():
    print("점검합니다")


schedule.every(10).minutes.do(job)

for _ in range(3):
    schedule.run_pending()
    time.sleep(1)

print("끝")
```

막히면 바로 위 `6.1 schedule — 파이썬 안에서 돈다` 설명을 다시 봅니다.


In [ ]:
import schedule
import time


def job():
    print("점검합니다")


schedule.every(10).minutes.do(job)

for _ in range(3):
    schedule.run_pending()
    time.sleep(1)

print("끝")


✅ `끝 — 10분이 안 지나서 job 은 한 번도 안 불린다`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-2 · 무엇이 보일까요</font></h3></td></tr></table>

이번에는 함수를 **직접** 부릅니다.

```python
def job():
    print("점검합니다")


job()
```

막히면 바로 위 `6.1 schedule — 파이썬 안에서 돈다` 설명을 다시 봅니다.


In [ ]:
def job():
    print("점검합니다")


job()


✅ `점검합니다`


**스케줄러는 시각을 기다릴 뿐입니다.** 하는 일은 우리가 만든 함수 그대로입니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-3 · 2초마다 돌려 보기</font></h3></td></tr></table>

`job` 을 **2초마다** 부르게 하고, 6초 동안 돌려 보시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `점검합니다` 가 두세 번 (기다린 시간에 따라 다릅니다) |

**💡 힌트**

1. `schedule.every(2).seconds.do(job)` 입니다.
2. `for _ in range(6):` 안에서 `run_pending()` 과 `time.sleep(1)` 을 부릅니다.
3. `do(job)` 에 괄호를 붙이지 않습니다 — `do(job())` 은 그 자리에서 실행됩니다.


In [ ]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-4 · 몇 번 돌았는지 세기</font></h3></td></tr></table>

`job` 이 **몇 번 불렸는지** 세어 마지막에 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `점검 N번 돌았습니다` |

**💡 힌트**

1. 함수 밖에 리스트를 하나 두고 `append` 하면 셀 수 있습니다.
2. 함수 안에서 숫자를 고치려면 `global` 이 필요한데, 리스트를 쓰면 그것 없이 됩니다.
3. 마지막에 `len()` 으로 셉니다.


In [ ]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-5 · 점검 일을 진짜 일로 바꾸기</font></h3></td></tr></table>

`job` 이 **`normalized_logs.json` 을 읽어 건수를 찍게** 고치시오. 스케줄러가 부르는 것이 이제 진짜 일입니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[점검] 정규화 16건` 이 몇 번 |

**💡 힌트**

1. `job` 함수 안에서 파일을 읽습니다.
2. 9/28에 배운 `json.load` 그대로입니다.
3. 준비 셀이 그 파일을 만들어 두었습니다.


In [ ]:
# 여기에 코드를 입력하세요


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 6-1 · 간격을 바꿔 보기</font></h3></td></tr></table>

간격을 `1`·`2`·`3` 초로 바꿔 가며 6초 동안 **몇 번 도는지** 각각 세어 보시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 간격이 짧을수록 많이 돈다 |

**💡 힌트**

1. 간격 셋을 리스트에 담고 `for` 로 돕니다.
2. 간격을 바꿀 때마다 `schedule.clear()` 로 앞 설정을 지웁니다.
3. **자주 돌수록 헛걸음이 는다** — 2교시의 폴링 이야기와 같습니다.


In [ ]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">6.2 `cron` 다섯 칸 · `scheduler_job.py` 조립</mark>

`schedule` 은 **그 프로그램이 떠 있어야** 돕니다. 창을 닫거나 컴퓨터가 꺼지면 멈춥니다.
운영체제에게 맡기면 꺼져 있어도 깨워 줍니다. 그것이 **`cron`** 입니다.

cron 은 빈칸으로 나뉜 **다섯 칸**으로 시각을 적습니다. 앞에서부터 **분 · 시 · 일 · 월 · 요일**입니다.

| 표현식 | 언제 |
|---|---|
| `0 6 * * *` | 매일 오전 6시 정각 |
| `*/10 * * * *` | 10분마다 |
| `0 9 * * 1` | 매주 월요일 오전 9시 |
| `0 0 1 * *` | 매달 1일 자정 |

읽는 요령이 있습니다. **별표가 아닌 칸만 읽고** 나머지는 「매번」으로 읽습니다.

> ⚠ **코랩에서는 `cron` 을 쓸 수 없습니다.** 런타임이 꺼지면 같이 사라지기 때문입니다.
> 오늘은 **읽고 쓰는 것만** 합니다. 실제로 거는 것은 10/8 에 내 컴퓨터에서 합니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-6 · 무엇이 보일까요</font></h3></td></tr></table>

cron 표현식을 읽는 연습입니다. 아래가 무엇을 뜻할지 적어 보세요.

```python
print("0 6 * * *")
print("*/10 * * * *")
print("0 9 * * 1")
```

막히면 바로 위 `6.2 cron 다섯 칸` 설명을 다시 봅니다.


In [ ]:
print("0 6 * * *")
print("*/10 * * * *")
print("0 9 * * 1")


✅ `매일 오전 6시 · 10분마다 · 매주 월요일 오전 9시`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-7 · 무엇이 보일까요</font></h3></td></tr></table>

다섯 칸을 쪼개 보면 이렇습니다.

```python
expr = "0 6 * * *"
parts = expr.split()

print("분", parts[0], "· 시", parts[1], "· 일", parts[2], "· 월", parts[3], "· 요일", parts[4])
```

막히면 바로 위 `6.2 cron 다섯 칸` 설명을 다시 봅니다.


In [ ]:
expr = "0 6 * * *"
parts = expr.split()

print("분", parts[0], "· 시", parts[1], "· 일", parts[2], "· 월", parts[3], "· 요일", parts[4])


✅ `분 0 · 시 6 · 일 * · 월 * · 요일 *`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-8 · 표현식을 읽어 설명하기</font></h3></td></tr></table>

표현식 셋을 받아 **분과 시가 무엇인지** 한 줄씩 출력하시오.

| | |
|---|---|
| 주어지는 값 | `exprs = ["0 6 * * *", "30 2 * * *", "*/5 * * * *"]` |
| 🎯 나와야 하는 결과 | `0 6 * * * → 분 0 시 6` 꼴로 세 줄 |

**💡 힌트**

1. `split()` 으로 다섯 칸을 나눕니다.
2. 0번이 분, 1번이 시입니다.
3. f-string 으로 원본과 함께 냅니다.


In [ ]:
exprs = ["0 6 * * *", "30 2 * * *", "*/5 * * * *"]


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-9 · 표현식을 직접 써 보기</font></h3></td></tr></table>

아래 셋을 cron 표현식으로 적으시오. 주석으로 답을 적으면 됩니다.

1. 매일 새벽 3시 정각
2. 1분마다
3. 매주 금요일 오후 6시

| | |
|---|---|
| 🎯 나와야 하는 결과 | `0 3 * * *` · `* * * * *` · `0 18 * * 5` |

**💡 힌트**

1. 칸 순서는 분·시·일·월·요일입니다.
2. 「매번」은 별표입니다.
3. 요일은 0 이 일요일입니다. 금요일은 5 입니다.


In [ ]:
# 여기에 답을 적습니다
# 1. 
# 2. 
# 3. 


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-10 · `scheduler_job.py` 만들기</font></h3></td></tr></table>

오후의 산출물 **`scheduler_job.py`** 를 만드시오. 새 문법은 없습니다.

0. 먼저 새 셀에서 **`!rm -f processed_ids.json`** 을 실행해 기억을 비웁니다 — 6교시에서 이미 `brute_force:admin` 을 적어 두었기 때문입니다.
1. `%%writefile scheduler_job.py` 로 시작합니다.
2. `--every` 인자로 **몇 초마다 돌릴지**를 받습니다. 기본값은 `2` 입니다.
3. `normalized_logs.json` 을 읽어 **룰 ①**(실패 3회 이상)을 돌립니다.
4. 걸린 경보를 **이미 보낸 것인지** `processed_ids.json` 으로 확인하고, 새 것만 화면에 찍습니다.
5. 보낸 번호를 파일에 저장합니다.
6. `schedule` 로 정해진 횟수만 돌고 끝냅니다.

| | |
|---|---|
| 🎯 1회차 | `[전송] brute_force:admin` |
| 🎯 2회차 | 아무것도 안 나간다 |

**💡 힌트**

1. 오전 4교시의 `argparse`, 6교시의 `load_done`·`save_done`, 9/29의 룰 ①을 잇는 것입니다.
2. `job` 함수 안에 전부 넣습니다.
3. 만든 뒤 `!python scheduler_job.py` 로 **두 번** 실행해 봅니다. 1회차와 2회차가 달라야 합니다.


In [ ]:
%%writefile scheduler_job.py


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-11 · 드라이브에 남기기</font></h3></td></tr></table>

오후 산출물 두 개를 내 드라이브 `agent_core` 폴더에 남기시오.

1. 아래 셀로 드라이브를 연결하고 `agent_core` 로 들어갑니다.
2. 문제 6-10(`scheduler_job.py`) 셀을 **다시 실행**합니다.
3. `processed_ids.json` 도 그 폴더에 생깁니다.

| | |
|---|---|
| 🎯 확인 | `scheduler_job.py` 와 `processed_ids.json` 이 드라이브에 있다 |

**💡 힌트**

1. 폴더를 옮기지 않으면 파일이 코랩 안에만 남습니다.
2. `normalized_logs.json` 도 같은 폴더에 있어야 실행됩니다.
3. 오전의 `webhook_server.py` 와 같은 폴더입니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 6-2 · 룰 세 개를 모두 돌리기</font></h3></td></tr></table>

`scheduler_job.py` 가 **룰 ①②③ 을 모두** 돌려 새 경보만 보내게 고치시오.

| | |
|---|---|
| 🎯 1회차 | 경보 세 건 |
| 🎯 2회차 | 0건 |

**💡 힌트**

1. 9/29 오후 문제 4-10 의 세 룰 코드를 `job` 안에 넣습니다.
2. 경보마다 `id` 를 만들어 `done` 과 견줍니다.
3. 룰마다 `id` 모양이 다릅니다 — 계정·IP·시각 중 무엇을 쓸지 정합니다.


In [ ]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#bbdefb; color:#1a1a1a; padding:6px 12px; border-radius:4px">6.정리 📋 한눈에</mark>

| 쓰는 법 | 뜻 |
|---|---|
| `schedule.every(10).minutes.do(job)` | 간격을 정한다. **괄호 없이** 함수 이름만 |
| `schedule.run_pending()` | 때가 됐는지 확인한다. **계속 불러야** 한다 |
| `cron` 다섯 칸 | 분 · 시 · 일 · 월 · 요일 |
| `0 6 * * *` | 매일 오전 6시 |

| | `schedule` | `cron` |
|---|---|---|
| 누가 깨우나 | 내 프로그램 | 운영체제 |
| 프로그램이 꺼지면 | 멈춘다 | 그래도 돈다 |
| 코랩에서 | 된다 | **안 된다** |

오후의 산출물은 드라이브 `agent_core` 폴더의 **`scheduler_job.py`** 와 **`processed_ids.json`** 입니다.


---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">8교시 (17:00–17:30) · 마무리</mark>


새로 배우는 것이 없습니다. 오늘 만든 것을 확인하고 마무리합니다.

### 8.1 오늘 만든 것 네 개

| 파일 | 무엇 | 언제 |
|---|---|---|
| `webhook_server.py` | 경보를 받는 창구 | 오전 4교시 |
| `test_webhook.sh` | 창구를 두드려 보는 확인용 | 오전 3교시 |
| `scheduler_job.py` | 정해진 때에 스스로 도는 점검 | 오후 7교시 |
| `processed_ids.json` | 이미 보낸 것을 적어 둔 기억 | 오후 6교시 |

### 8.2 오늘 못 한 것 — `cron`

`cron` 은 **코랩에서 쓸 수 없습니다.** 런타임이 꺼지면 같이 사라지기 때문입니다.
표현식을 읽고 쓰는 것까지 했고, **실제로 거는 것은 10/8** 에 내 컴퓨터에서 합니다.

### 8.3 일곱 날이 한 줄로

| 날 | 무엇을 더했나 |
|---|---|
| 9/22 · 9/23 | 값을 담고 파일을 읽는다 |
| 9/28 | 깨져도 멈추지 않고 기록한다 |
| 9/29 | 줄글에서 꺼내 수상한 것을 고른다 |
| 9/30 | 밖에 물어본다 |
| **10/2** | **받는 문을 열고 스스로 돈다** |

### 8.4 다음 주 예고

10/6 은 **LLM 과 에이전트**입니다. 오늘 만든 **트리거 · 조건 · 액션** 세 칸에서
**조건** 자리를 사람이 적은 규칙 대신 **모델이 판단**하게 바꾸는 것이 그날의 이야기입니다.


---

# <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">정답 · 먼저 풀어 본 뒤에 엽니다</mark>

각 셀은 제목만 보입니다. 「코드 표시」를 누르면 코드가 열립니다.


In [ ]:
#@title 정답 4-3 { display-mode: "form" }
events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

for event in events:
    print(event["rule"])


In [ ]:
#@title 정답 4-4 { display-mode: "form" }
events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

for event in events:
    if event["rule"] != "night_login":
        print(event["rule"])


In [ ]:
#@title 정답 4-5 { display-mode: "form" }
events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

for event in events:                          # 트리거 — 경보 목록이 있다
    if event["rule"] == "brute_force":        # 조건 — 브루트포스인가
        print("[알림]", event["user"])         # 액션 — 알린다


In [ ]:
#@title 정답 ⭐4-1 { display-mode: "form" }
events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

for event in events:
    if event["rule"] == "night_login":
        print("[알림]", event["user"])


In [ ]:
#@title 정답 4-8 { display-mode: "form" }
import requests


def send_alert(event):
    response = requests.post("http://127.0.0.1:5005/webhook", json=event, timeout=5)
    return response.json()["count"]


print(send_alert({"rule": "brute_force", "user": "admin"}))


In [ ]:
#@title 정답 4-9 { display-mode: "form" }
import requests

events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]


def send_alert(event):
    response = requests.post("http://127.0.0.1:5005/webhook", json=event, timeout=5)
    return response.json()["status"]


for event in events:
    result = send_alert(event)
    print(f"[전송] {event['rule']} → {result}")


In [ ]:
#@title 정답 4-10 { display-mode: "form" }
import json
import os

if os.path.exists("received_alerts.json"):
    with open("received_alerts.json", encoding="utf-8") as f:
        rows = json.load(f)
    print(f"받은 경보 {len(rows)}건")
    for row in rows:
        print(" ", row["rule"])
else:
    print("아직 받은 것이 없습니다 — 문제 4-9 를 먼저 실행하세요")


In [ ]:
#@title 정답 ⭐4-2 { display-mode: "form" }
import requests


def send_alert(event, port=5005):
    try:
        response = requests.post(f"http://127.0.0.1:{port}/webhook", json=event, timeout=5)
        return response.json()["status"]
    except requests.RequestException:
        return None


result = send_alert({"rule": "brute_force"}, port=59999)   # 아무도 안 듣는 포트

if result:
    print("전송 성공:", result)
else:
    print("전송 실패 — 다음 줄은 그대로 이어집니다")

print("프로그램은 살아 있습니다")


In [ ]:
#@title 정답 5-3 { display-mode: "form" }
events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

for turn in [1, 2]:
    for event in events:
        print(f"{turn}회차 [전송] {event['rule']}")


In [ ]:
#@title 정답 5-4 { display-mode: "form" }
events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

for event in events:
    print(event["id"])


In [ ]:
#@title 정답 5-5 { display-mode: "form" }
events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

done = ["brute_force:admin"]

for event in events:
    if event["id"] not in done:
        print(event["id"])


In [ ]:
#@title 정답 ⭐5-1 { display-mode: "form" }
events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

for event in events:
    if "user" in event:
        event_id = f"{event['rule']}:{event['user']}"
    else:
        event_id = f"{event['rule']}:{event['ip']}"
    print(event_id)


In [ ]:
#@title 정답 5-8 { display-mode: "form" }
import json
import os


def load_done():
    if os.path.exists("processed_ids.json"):
        with open("processed_ids.json", encoding="utf-8") as f:
            return json.load(f)
    return []


def save_done(done):
    with open("processed_ids.json", "w", encoding="utf-8") as f:
        json.dump(done, f, ensure_ascii=False, indent=2)

events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

done = load_done()
sent = 0

for event in events:
    if event["id"] not in done:
        print("[전송]", event["rule"])
        done.append(event["id"])
        sent = sent + 1

save_done(done)

print("보낸 건수", sent)


In [ ]:
#@title 정답 5-9 { display-mode: "form" }
print("문제 5-8 의 셀을 다시 실행하면 됩니다.")
print("앞 실행이 processed_ids.json 에 세 건을 적어 두었으므로 이번에는 0 건입니다.")


In [ ]:
#@title 정답 5-10 { display-mode: "form" }
import json
import os


def load_done():
    if os.path.exists("processed_ids.json"):
        with open("processed_ids.json", encoding="utf-8") as f:
            return json.load(f)
    return []


def save_done(done):
    with open("processed_ids.json", "w", encoding="utf-8") as f:
        json.dump(done, f, ensure_ascii=False, indent=2)

done = load_done()

print(f"이미 처리한 사건 {len(done)}건")
for event_id in done:
    print(" ", event_id)


In [ ]:
#@title 정답 5-11 { display-mode: "form" }
import json
import os


def load_done():
    if os.path.exists("processed_ids.json"):
        with open("processed_ids.json", encoding="utf-8") as f:
            return json.load(f)
    return []


def save_done(done):
    with open("processed_ids.json", "w", encoding="utf-8") as f:
        json.dump(done, f, ensure_ascii=False, indent=2)

events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

events.append({"id": "brute_force:guest", "rule": "brute_force", "user": "guest", "count": 3})

done = load_done()
sent = 0

for event in events:
    if event["id"] not in done:
        print("[전송]", event["id"])
        done.append(event["id"])
        sent = sent + 1

save_done(done)

print("보낸 건수", sent)


In [ ]:
#@title 정답 ⭐5-2 { display-mode: "form" }
print("!rm -f processed_ids.json     # 기억을 지운다")
print("그다음 문제 5-8 을 다시 돌리면 세 건이 모두 다시 나갑니다.")
print()
print("멱등성은 코드가 아니라 이 파일이 지키고 있습니다.")


In [ ]:
#@title 정답 6-3 { display-mode: "form" }
import schedule
import time


def job():
    print("점검합니다")


schedule.every(2).seconds.do(job)

for _ in range(6):
    schedule.run_pending()
    time.sleep(1)

print("끝")


In [ ]:
#@title 정답 6-4 { display-mode: "form" }
import schedule
import time

runs = []


def job():
    runs.append(1)


schedule.every(2).seconds.do(job)

for _ in range(6):
    schedule.run_pending()
    time.sleep(1)

print(f"점검 {len(runs)}번 돌았습니다")


In [ ]:
#@title 정답 6-5 { display-mode: "form" }
import schedule
import time
import json


def job():
    with open("normalized_logs.json", encoding="utf-8") as f:
        rows = json.load(f)
    print(f"[점검] 정규화 {len(rows)}건")


schedule.every(2).seconds.do(job)

for _ in range(6):
    schedule.run_pending()
    time.sleep(1)

print("끝")


In [ ]:
#@title 정답 ⭐6-1 { display-mode: "form" }
import schedule
import time

for every in [1, 2, 3]:
    schedule.clear()
    runs = []
    schedule.every(every).seconds.do(lambda: runs.append(1))
    for _ in range(6):
        schedule.run_pending()
        time.sleep(1)
    print(f"{every}초마다 — {len(runs)}번 돌았습니다")


In [ ]:
#@title 정답 6-8 { display-mode: "form" }
exprs = ["0 6 * * *", "30 2 * * *", "*/5 * * * *"]

for expr in exprs:
    parts = expr.split()
    print(f"{expr} → 분 {parts[0]} 시 {parts[1]}")


In [ ]:
#@title 정답 6-9 { display-mode: "form" }
print("1. 0 3 * * *      매일 새벽 3시 정각")
print("2. * * * * *      1분마다")
print("3. 0 18 * * 5     매주 금요일 오후 6시")


In [ ]:
#@title 정답 6-10 { display-mode: "form" }
code = """import argparse
import json
import os
import schedule
import time

parser = argparse.ArgumentParser(description="정기 점검 작업")
parser.add_argument("--every", type=int, default=2, help="몇 초마다 돌릴지")
args = parser.parse_args()

THRESHOLD = 3


def load_done():
    if os.path.exists("processed_ids.json"):
        with open("processed_ids.json", encoding="utf-8") as f:
            return json.load(f)
    return []


def job():
    with open("normalized_logs.json", encoding="utf-8") as f:
        rows = json.load(f)

    count = {}
    for row in rows:
        if row["level"] == "WARN":
            user = row["user"]
            if user in count:
                count[user] = count[user] + 1
            else:
                count[user] = 1

    done = load_done()
    sent = 0

    for user in count:
        if count[user] >= THRESHOLD:
            event_id = f"brute_force:{user}"
            if event_id not in done:
                print("[전송]", event_id)
                done.append(event_id)
                sent = sent + 1

    with open("processed_ids.json", "w", encoding="utf-8") as f:
        json.dump(done, f, ensure_ascii=False, indent=2)

    print(f"점검 완료 — 새 경보 {sent}건")


schedule.every(args.every).seconds.do(job)

for _ in range(4):
    schedule.run_pending()
    time.sleep(1)
"""

with open("scheduler_job.py", "w", encoding="utf-8") as f:
    f.write(code)

import os

if os.path.exists("processed_ids.json"):
    os.remove("processed_ids.json")
    print("processed_ids.json 을 비웠습니다 — 6교시 기록이 남아 있으면 1회차가 0건이 됩니다")

print("scheduler_job.py 를 만들었습니다. !python scheduler_job.py 로 두 번 실행해 보세요.")


In [ ]:
#@title 정답 6-11 { display-mode: "form" }
print("!mkdir -p /content/drive/MyDrive/agent_core")
print("%cd /content/drive/MyDrive/agent_core")
print("그다음 준비 셀과 scheduler_job.py 셀을 다시 실행합니다.")


In [ ]:
#@title 정답 ⭐6-2 { display-mode: "form" }
print("job() 안에서 세 룰을 차례로 돌리고, 경보마다 id 를 만들어 걸러 냅니다.")
print()
print('룰 ① → f"brute_force:{user}"')
print('룰 ② → f"password_spraying:{ip}"')
print('룰 ③ → f"night_login:{user}:{time}"')
print()
print("id 를 만드는 규칙만 정하면 나머지는 6-10 과 같습니다.")
